# Week 37, Day 3: EAFP and LBYL

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w037d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def lbyl(text):
    if text.isdigit():
        return int(text)
    return None

def eafp(text):
    try:
        return int(text)
    except ValueError:
        return None

for text in ["42", " 42", "-3", "²"]:
    try:
        print(repr(text), lbyl(text), eafp(text))
    except ValueError:
        print(repr(text), "lbyl crashed!", eafp(text))

In [ ]:
stock = {"tea": 4, "rice": 0}

# LBYL
count = stock["milk"] if "milk" in stock else 0
# EAFP
try:
    count = stock["milk"]
except KeyError:
    count = 0
# neither: the default is built in
count = stock.get("milk", 0)
print(count)

words = ["kettle", "apple", "Ada"]
print(next((w for w in words if w[0] in "aeiouAEIOU"), "none"))      # first match, or a default
print("key=value=x".partition("="), "novalue".partition("="))           # never fails, unlike split

In [ ]:
import os

def lbyl_read(path, meanwhile):
    if os.path.exists(path):
        meanwhile()                      # something else runs between the check and the use
        with open(path) as f:
            return f.read()
    return "no file"

def eafp_read(path, meanwhile):
    meanwhile()
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        return "no file"

def another_program():
    os.remove("note.txt")

for reader in [eafp_read, lbyl_read]:
    with open("note.txt", "w") as f:
        f.write("hello")
    try:
        print(reader.__name__, "->", reader("note.txt", another_program))
    except FileNotFoundError as err:
        print(reader.__name__, "-> crashed:", type(err).__name__)

In [ ]:
import os

if os.path.exists("draft.txt"):
    os.remove("draft.txt")
for attempt in [1, 2]:
    try:
        with open("draft.txt", "x") as f:
            f.write(f"attempt {attempt}")
        print("created on attempt", attempt)
    except FileExistsError:
        print("attempt", attempt, "refused: draft.txt already exists")
with open("draft.txt") as f:
    print(f.read())

In [ ]:
prices = {"tea": 300, "rice": 1200}
discounts = {"rice": 200}

def price_of(item):
    try:
        price = prices[item]
    except KeyError:
        return f"{item}: not sold here"
    else:
        return f"{item}: {price - discounts.get(item, 0)}"

for item in ["tea", "rice", "milk"]:
    print(price_of(item))

In [ ]:
import os
from contextlib import suppress

with open("old.log", "w") as f:
    f.write("x")
for name in ["old.log", "old.log", "never-existed.log"]:
    with suppress(FileNotFoundError):
        os.remove(name)
        print("removed", name)
print(os.path.exists("old.log"))

In [ ]:
def transfer(accounts, source, target, amount):
    if source not in accounts:
        raise KeyError(f"no account {source!r}")
    if target not in accounts:
        raise KeyError(f"no account {target!r}")
    if not 0 < amount <= accounts[source]:
        raise ValueError(f"can't move {amount} from a balance of {accounts[source]}")
    accounts[source] -= amount
    accounts[target] += amount

accounts = {"ada": 100, "bo": 20}
for args in [("ada", "bo", 30), ("ada", "chi", 10), ("bo", "ada", 500)]:
    try:
        transfer(accounts, *args)
    except (KeyError, ValueError) as err:
        print("refused:", err)
print(accounts)

> **Key idea.** LBYL checks first; EAFP tries and catches. A check is a second copy of the operation's rule and can disagree with it (`isdigit` against `int`), and the world can change between check and use (a race: the file is deleted after `exists` said yes), so for operations with their own rules, let the operation decide. Prefer built-in defaults (`dict.get`, `getattr`, `next(it, default)`, `partition`), keep each `try` narrow with `else`, and use `suppress` for errors that truly don't matter. Open with `"x"` to create only if absent. Check first for your own rules, when failure is normal, and before changes that mustn't stop halfway.

### Exercise: Asking int

Write `to_int(text, default=None)`, which returns `int(text)`, or `default` if `int` can't convert it. Let `int` decide (no `isdigit`), and don't catch anything except `ValueError`.

In [ ]:
def to_int(text, default=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_to_int():\n    \"int decides\"\n    assert to_int(\"42\") == 42 and to_int(\" 42 \") == 42 and to_int(\"-3\") == -3 and to_int(\"1_000\") == 1000\n    assert to_int(\"\u00b2\") is None and to_int(\"4.5\", 0) == 0 and to_int(\"\") is None\n    assert \"isdigit\" not in source\n    try:\n        to_int(None)\n        assert False, \"only ValueError should be caught: None is a bug, a TypeError\"\n    except TypeError:\n        pass")

### Exercise: Digging into data

Write `dig(data, path, default=None)`, which follows a path of keys and indexes into nested dictionaries and lists, and returns the value, or `default` if any step isn't there. `path` is a list such as `["users", 0, "name"]`. Use EAFP: catch `LookupError` (a missing key or index) and `TypeError` (indexing into something that isn't a container, like a number), and nothing else.

In [ ]:
def dig(data, path, default=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dig():\n    \"digging\"\n    data = {\"users\": [{\"name\": \"Ada\", \"tags\": [\"x\"]}, {\"name\": \"Bo\"}], \"count\": 2}\n    assert dig(data, [\"users\", 0, \"name\"]) == \"Ada\" and dig(data, [\"users\", -1, \"name\"]) == \"Bo\"\n    assert dig(data, [\"users\", 1, \"tags\", 0], \"none\") == \"none\"\n    assert dig(data, [\"users\", 5]) is None and dig(data, [\"count\", 0], \"no\") == \"no\"\n    assert dig(data, []) == data")

### Exercise: Tidying up

Write `remove_all(paths)`, which deletes each file in the list with `os.remove`, ignoring files that are already gone (use `contextlib.suppress`), and returns how many it actually removed.

In [ ]:
import os

def remove_all(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_remove():\n    \"removing\"\n    write_file(\"a.tmp\", \"1\")\n    write_file(\"b.tmp\", \"2\")\n    assert remove_all([\"a.tmp\", \"gone.tmp\", \"b.tmp\", \"a.tmp\"]) == 2\n    assert \"a.tmp\" not in files_now() and \"b.tmp\" not in files_now()\n    assert remove_all([]) == 0 and \"suppress\" in source")

### Exercise: All or nothing

Write `move_stock(stock, moves)`. `stock` maps item names to counts; `moves` is a list of `(item, change)` pairs, where a change can be negative. Apply all the moves, or none: first check every move (an unknown item raises `KeyError`, and any count that would go below 0 raises `ValueError`), and only then change `stock`. Several moves can affect the same item, so check the running totals.

In [ ]:
def move_stock(stock, moves):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_moves():\n    \"all or nothing\"\n    stock = {\"tea\": 5, \"rice\": 2}\n    move_stock(stock, [(\"tea\", -2), (\"rice\", 3)])\n    assert stock == {\"tea\": 3, \"rice\": 5}\n    for moves, error in [([(\"tea\", -1), (\"milk\", 1)], KeyError), ([(\"rice\", 1), (\"tea\", -2), (\"tea\", -2)], ValueError)]:\n        try:\n            move_stock(stock, moves)\n            assert False, f\"{moves} should raise {error.__name__}\"\n        except error:\n            pass\n        assert stock == {\"tea\": 3, \"rice\": 5}, f\"nothing should change: {stock}\"\n    move_stock(stock, [(\"tea\", -3), (\"tea\", 1)])\n    assert stock == {\"tea\": 1, \"rice\": 5}")

### Bug hunt: A try too wide

Tea has no discount, but the missing discount raises `KeyError` inside the `try`, so tea is reported as `not sold here`. Fix it so the output is `tea: 300`, then `rice: 1000`, then `milk: not sold here`.

In [ ]:
prices = {"tea": 300, "rice": 1200}
discounts = {"rice": 200}

for item in ["tea", "rice", "milk"]:
    try:
        print(f"{item}: {prices[item] - discounts[item]}")
    except KeyError:
        print(f"{item}: not sold here")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a narrow try\"\n    expect_output(\"tea: 300\\nrice: 1000\\nmilk: not sold here\")")

### Bug hunt: The check that disagrees

`parse_temps` should accept negative temperatures, but its `isdigit` check rejects them, so `-4` is reported as bad. Let `int` decide, so the output is `[12, -4, 0]` then `bad: 'warm'`.

In [ ]:
def parse_temps(texts):
    temps = []
    for text in texts:
        if not text.isdigit():
            raise ValueError(f"bad: {text!r}")
        temps.append(int(text))
    return temps

print(parse_temps(["12", "-4", "0"]))
try:
    parse_temps(["12", "warm"])
except ValueError as err:
    print(err)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"int decides\"\n    expect_output(\"[12, -4, 0]\\nbad: 'warm'\")")

### Bug hunt: Two errors in one clause

A `0` should print `can't share between 0`, but because the division is inside the same `try` as the conversion, it's reported as `not a number`. Fix it so the output is `4`, `not a number`, `can't share between 0`.

In [ ]:
for raw in ["3", "x", "0"]:
    try:
        n = int(raw)
        print(12 // n)
    except (ValueError, ZeroDivisionError):
        print("not a number")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one clause per problem\"\n    expect_output(\"4\\nnot a number\\ncan't share between 0\")")

### Puzzle: Create, don't overwrite

Arrange the lines into a function that creates a file only if it doesn't exist yet, with no gap between checking and creating. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
stock = {"tea": 0}

a = stock.get("milk", 5)
b = getattr("text", "upper", None) is not None
c = next((n for n in [1, 3, 4, 6] if n % 2 == 0), None)
d = "a=b=c".partition("=")[2]
e = "-7".isdigit()

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # d
t = ___   # e

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q is True and r == 4, \"p, q and r\"\n    assert s == \"b=c\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: saving without overwriting

Write `save_new(stem, ext, text)`, which saves `text` to a **new** file and returns the name it used. It tries `stem.ext` first, then `stem-1.ext`, `stem-2.ext` and so on, opening each with mode `"x"` (no `exists` checks: another program could create a file at any moment). After 100 numbered attempts all fail (up to `stem-100.ext`), it raises `FileExistsError(f"no free name for {stem}.{ext}")`. Check the arguments first, before touching any file: `text` must be a string (else `TypeError`), and `stem` must be non-empty and contain no `/` (else `ValueError`).

In [ ]:
def save_new(stem, ext, text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_save():\n    \"saving\"\n    reset_files()\n    assert save_new(\"notes\", \"txt\", \"a\") == \"notes.txt\"\n    assert save_new(\"notes\", \"txt\", \"b\") == \"notes-1.txt\"\n    write_file(\"notes-2.txt\", \"someone else's\")\n    assert save_new(\"notes\", \"txt\", \"c\") == \"notes-3.txt\"\n    assert read_file(\"notes.txt\") == \"a\" and read_file(\"notes-1.txt\") == \"b\" and read_file(\"notes-2.txt\") == \"someone else's\" and read_file(\"notes-3.txt\") == \"c\"\n    assert \"exists(\" not in source\ndef test_full():\n    \"no free name\"\n    reset_files()\n    for i in range(101):\n        write_file(\"x.md\" if i == 0 else f\"x-{i}.md\", \"\")\n    try:\n        save_new(\"x\", \"md\", \"hi\")\n        assert False\n    except FileExistsError as err:\n        assert str(err) == \"no free name for x.md\"\ndef test_checks():\n    \"checking first\"\n    reset_files()\n    for args, error in [((\"a\", \"txt\", 5), TypeError), ((\"\", \"txt\", \"x\"), ValueError), ((\"a/b\", \"txt\", \"x\"), ValueError)]:\n        try:\n            save_new(*args)\n            assert False, f\"{args} should raise {error.__name__}\"\n        except error:\n            pass\n    assert files_now() == [], \"nothing should be created\"")

## Warm-up

Neither version is "the safe one". The check `isdigit()` turned away `" 42"` and `"-3"`, which `int` accepts, and let through `"²"`, which `int` rejects, so `lbyl("²")` crashed. The check and the operation it guards follow **different rules**. That's the first of two ways checking first goes wrong.

## Two styles, again

| | Look before you leap (LBYL) | Easier to ask forgiveness than permission (EAFP) |
|---|---|---|
| Shape | `if it will work: do it` | `try: do it` / `except: handle it` |
| The rule used | your check's | the operation's own |
| Cost when it works | the check, then the work | just the work |
| Cost when it fails | just the check | the work so far, plus raising and catching |
| Danger | the check and the use disagree, or the world changes in between | the `try` catches more than you meant |

The saying behind EAFP, "it's easier to ask forgiveness than it is to get permission", is usually credited to Grace Hopper, and Python's own glossary describes the style as common in Python. In Python, raising and catching an exception is cheap enough that EAFP is the usual choice when failure is **unusual**. When failure is the **normal** case (most keys missing, most lines blank), a check reads better and does less work.

Often there's a third way, better than both: a tool that **builds the default in**.

`dict.get`, `getattr(obj, name, default)`, `next(iterator, default)` and `str.partition` all handle "it might not be there" without an `if` or a `try`. Reach for them first.

## When the check disagrees

The warm-up's problem is common: a check is a **second copy** of the rule, and copies drift. `int` accepts spaces around the number, a sign, underscores between digits (`"1_000"`), and digits from other writing systems (`"٣"`, Arabic-Indic three), and it rejects superscripts, which `isdigit` counts as digits. Your check would have to repeat all of that exactly. Asking `int` itself is the only reliable way to know whether `int` will work.

The same goes for other checks:

| Check first | Why it can be wrong |
|---|---|
| `if os.path.exists(path): open(path)` | the file can exist but be unreadable (`PermissionError`), or be a folder |
| `if hasattr(obj, "read"): obj.read()` | `read` might need arguments, or fail for its own reasons |
| `if len(items) > i: items[i]` | forgets that negative `i` works for lists |
| `if "@" in email:` | says nothing about whether the address works |

When the rule belongs to an operation, let the operation decide, and catch its error. When the rule is **yours** (an age must be 0 to 130, a name can't be empty), there's no operation to ask, and checking is the only way: that's validation, and it's tomorrow's subject.

## When the world changes

The second problem is **time**. Between your check and your use, something else can change the world: another program, another user, another part of your own program. This is called a race condition, and this particular one has a name, "time of check to time of use":

Here the "other program" is a function so you can see it happen every time. In real life it happens rarely, which is worse: a bug that appears once a month, on a busy server, is very hard to find. The check didn't make the code safe; it only made the failure rarer. The EAFP version has no gap, because the operation **is** the check.

Files offer a neat example of letting the operation check. To create a file **only if it doesn't already exist**, don't check with `exists` and then open with `"w"` (which overwrites). Open with mode `"x"`, which refuses if the file is there:

## Narrow tries and quiet helpers

EAFP's danger is catching too much. Week 30 kept the `try` around only the risky line; with `else`, you can do that even when more work follows:

If `prices[item] - discounts[item]` had been inside the `try`, a missing **discount** would also raise `KeyError`, and tea would be reported as "not sold here". A wide `try` doesn't only catch the error you expected; it catches the same **type** of error from anywhere inside it.

When you want to **ignore** one kind of error completely, `contextlib.suppress` says so in one line:

`with suppress(FileNotFoundError):` is the same as `try: ... except FileNotFoundError: pass`, and reads as a decision rather than an accident. (It's built with `with`, which Phase 5 explains.) Use it only for errors that truly don't matter: here, "it's already gone" is exactly what we wanted.

## When to look first

Checking first is the better choice when:

- **the rule is yours**, not an operation's: a quantity above 0, a date in the future. There's no operation to ask.
- **failure is normal**: if most dictionary keys are missing, `if key in d` reads better than a `try` that usually fails.
- **you mustn't stop halfway**. If an operation makes several changes, a failure in the middle can leave things half-done. Check **everything** first, then make the changes:

Without the checks at the top, `transfer(accounts, "ada", "chi", 10)` would take 10 from Ada and **then** fail on Chi, and the money would vanish. Validating before changing anything keeps the data consistent. Databases call this property atomicity: all of it happens or none of it does.

A good rule: **LBYL for your own rules and for anything that changes several things; EAFP for operations that know their own rules, and for anything another program could change.**

## Practice

## Quiz

## Daily challenge

## Before you go

Find an `if os.path.exists` or an `isdigit` check in your own code. Is it guarding an operation that could decide for itself? Tomorrow: the `logging` module, so your programs can say what they're doing without `print`.